### Day 10 Assignment: Databricks SQL, AI/BI Dashboards & Genie Agents

### Basic Tasks 


#### 1. Create a Serverless SQL Warehouse & Run 3 Exploratory Queries

In [0]:
-- Query 1 — Revenue by Region
select 
  round(sum(sale_amount),2) as total_revenue, 
  region
from dev.gold.sales_summary
group by region
order by total_revenue desc;


In [0]:
-- Query 2 — Monthly Sales Trend
select 
    sale_month as month, 
    round(sum(sale_amount),2) as total_revenue
from dev.gold.sales_summary
group by sale_month
order by month;


In [0]:
-- Query 3 — Top 10 Products
select 
    product_id, 
    product_name, 
    round(sum(sale_amount),2) as total_revenue
from dev.gold.sales_summary
group by product_id, product_name
order by total_revenue desc
limit 10;

#### 2. Build the AI/BI Dashboard
Created an AI/BI dashboard using the Gold sales table created in the previous days.


#### 3. Publish the Dashboard and Test Ask Genie

After creating the visualizations, publish the AI/BI dashboard so that it can be viewed by the intended users.

Once the dashboard is published, use the Ask Genie feature to test natural-language questions against the dashboard data.

![image_1789044975193.png](./image_1789044975193.png "image_1789044975193.png")

### Intermediate Tasks 

#### 4. Add a Dashboard Filter

![image_1789046111507.png](./image_1789046111507.png "image_1789046111507.png")

#### 5. Set up a Genie Agent

![image_1789047126316.png](./image_1789047126316.png "image_1789047126316.png")

#### 6. Genie Agent Accuracy Improvement

##### Initial Question

**Question:**
What are our sales?

##### Before Improvement

Initially, Genie interpreted "sales" as the number of units sold and generated logic equivalent to:

```sql
SELECT SUM(quantity)
FROM <catalog>.<schema>.gold_sales;
```

This was incorrect because the business definition of sales/revenue in our Gold table is based on `sales_amount`, while `quantity` represents units sold.

##### Improvement

I added the following instruction to the Genie Agent:

> "Sales, revenue, and total sales mean SUM(sales_amount). Never use SUM(quantity) or SUM(unit_price) when calculating sales or revenue. Use SUM(quantity) only when the user explicitly asks for units sold."

I also added a trusted query for total revenue:

```sql
SELECT
    SUM(sales_amount) AS total_revenue
FROM <catalog>.<schema>.gold_sales;
```

The trusted query was associated with questions such as:

* What are our total sales?
* What is our total revenue?
* How much did we sell?
* What were our sales?

##### After Improvement

I asked Genie the same question again:

> What are our sales?

Genie now interpreted "sales" according to the defined business rule and used `SUM(sales_amount)`.

##### Result

**Before:** "Sales" was interpreted as units sold (`SUM(quantity)`).

**After:** "Sales" was correctly interpreted as revenue (`SUM(sales_amount)`).

This demonstrated how Genie Agent instructions and trusted assets can improve natural-language-to-SQL accuracy and enforce business-specific definitions.


### Advanced Tasks 


#### 7. Genie Agent Curation Checklist for Cyntexa

##### Objective

Before using a Genie Agent for executive-facing questions, ensure that the underlying Gold data is well documented, governed, and easy for Genie to understand.

##### Curation Checklist

##### Unity Catalog Metadata

- **Table descriptions:** Explain the business purpose of each table and what one row represents.
- **Column comments:** Clearly describe important columns such as `sales_amount`, `quantity`, `region`, and `sale_date`.
- **Business definitions:** Define ambiguous terms such as "sales", "revenue", and "units sold".
- **Date definitions:** Specify which date column should be used for monthly, quarterly, or yearly analysis.
- **Data ownership:** Expose only approved Gold tables rather than raw, temporary, or deprecated tables.

##### Governance

Use Unity Catalog permissions, views, or masking to ensure Genie only exposes data the user is authorized to access, especially sensitive customer attributes.

Add a Genie instruction such as:

- Sales/revenue = SUM(sales_amount).
- Units sold = SUM(quantity).
- Use sale_date for time-based sales analysis.

In [0]:
-- Minimal Practical Setup
-- Example table description:
COMMENT ON TABLE dev.gold.sales_summary IS
'Approved Gold sales table. Each row represents one completed sales transaction.';


#### 8. Photon, Predictive I/O, and Intelligent Workload Management

A Serverless SQL Warehouse can handle ad hoc dashboard queries efficiently because Databricks uses different optimizations for different performance needs.

**Photon** speeds up SQL query execution using an optimized query engine.

**Predictive I/O** improves data-reading efficiency and reduces unnecessary I/O during queries.

**Intelligent Workload Management (IWM)** dynamically manages resources and concurrent workloads to maintain performance when multiple users run queries.

##### Why Serverless?

AI/BI dashboards usually have **ad hoc and unpredictable queries**, with multiple users potentially querying at the same time. Serverless is a good fit because Databricks manages the underlying compute and automatically handles changing workloads.

Overall, Photon improves query execution, Predictive I/O improves data access, and IWM manages concurrent workloads, resulting in a fast and responsive dashboard experience.

For this use case, I would prefer **Serverless SQL Warehouse** over Classic/Pro because it reduces infrastructure management and is better suited for unpredictable, interactive BI workloads.

#### 9. Executive Revenue Dashboard

I built an executive dashboard to answer the business question:

**"Are we on track to hit our quarterly revenue target by region?"**

The dashboard uses the Gold sales and target data and includes three key visualizations:

1. **Actual vs Target Revenue** – compares actual quarterly revenue with the target.
2. **Revenue Performance by Region** – shows how each region is performing against its target.
3. **Quarterly Revenue Trend** – shows actual revenue and target performance over time.

The dashboard also calculates **Target Achievement %** as:

`Actual Revenue / Target Revenue × 100`

The dashboard was published and tested using **Ask Genie**. Genie correctly identified that the regions were achieving approximately **90.9% of their quarterly targets**, with revenue consistently below target.

For a VP, I would provide these instructions:

> "Use this dashboard to understand quarterly revenue performance by region. Compare actual revenue with the corresponding target, show the percentage of target achieved, and highlight regions that are above or below target. Explain the results in simple business terms without using technical table or column names."

![image_1789150336989.png](./image_1789150336989.png "image_1789150336989.png")